## 1. Base de Librerías

In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuración
pd.set_option('display.max_columns', None)

## 2. Carga de Datasets

In [12]:
# Cargar dataset con target
df_target = pd.read_csv('../../OPERA_POSTQX_con_target.csv')

# Verificar y renombrar columna de merge si es necesario
if 'Documento PMD (valoración preanestésica)' in df_target.columns:
    df_target = df_target.rename(columns={'Documento PMD (valoración preanestésica)': 'Documento PMD'})

print("Dataset Target:")
print(f"  Filas: {df_target.shape[0]:,} × Columnas: {df_target.shape[1]}")
print(f"  Columnas: {list(df_target.columns)[:5]}...")

Dataset Target:
  Filas: 29,865 × Columnas: 138
  Columnas: ['Documento PMD', 'Número de episodio (valoración preanestésica)', 'Prestacion actual consulta anestesia', 'Preadmision anestesia', 'Orden']...


In [13]:
# Cargar datasets con anomalías
df_anomalias_completo = pd.read_csv('../../OPERA_BASE_clean_con_anomalias.csv')
df_anomalias_pediatricos = pd.read_csv('../../OPERA_BASE_clean_con_anomalias_PEDIATRICOS.csv')
df_anomalias_adultos = pd.read_csv('../../OPERA_BASE_clean_con_anomalias_ADULTOS.csv')

print("Datasets con Anomalías:")
print(f"  Completo: {df_anomalias_completo.shape[0]:,} filas × {df_anomalias_completo.shape[1]} columnas")
print(f"  Pediátricos: {df_anomalias_pediatricos.shape[0]:,} filas × {df_anomalias_pediatricos.shape[1]} columnas")
print(f"  Adultos: {df_anomalias_adultos.shape[0]:,} filas × {df_anomalias_adultos.shape[1]} columnas")

Datasets con Anomalías:
  Completo: 30,962 filas × 222 columnas
  Pediátricos: 6,683 filas × 222 columnas
  Adultos: 24,279 filas × 222 columnas


## 3. Preparación de Datasets

In [14]:
# Renombrar columnas de anomalías para identificar origen
df_anomalias_completo = df_anomalias_completo.rename(columns={'es_anomalia': 'es_anomalia_completo'})
df_anomalias_pediatricos = df_anomalias_pediatricos.rename(columns={'es_anomalia': 'es_anomalia_pediatricos'})
df_anomalias_adultos = df_anomalias_adultos.rename(columns={'es_anomalia': 'es_anomalia_adultos'})

print("Columnas de anomalías renombradas:")
print("  - es_anomalia_completo")
print("  - es_anomalia_pediatricos")
print("  - es_anomalia_adultos")

Columnas de anomalías renombradas:
  - es_anomalia_completo
  - es_anomalia_pediatricos
  - es_anomalia_adultos


## 4. Merge de Datasets

In [15]:
# Merge con dataset completo
df_merged_completo = pd.merge(
    df_anomalias_completo[['Documento PMD', 'es_anomalia_completo']],
    df_target[['Documento PMD', 'target']],
    on='Documento PMD',
    how='inner'
)

print(f"Merge completo: {df_merged_completo.shape[0]:,} filas")

Merge completo: 29,865 filas


In [16]:
# Merge con dataset pediátricos
df_merged_pediatricos = pd.merge(
    df_anomalias_pediatricos[['Documento PMD', 'es_anomalia_pediatricos']],
    df_target[['Documento PMD', 'target']],
    on='Documento PMD',
    how='inner'
)

print(f"Merge pediátricos: {df_merged_pediatricos.shape[0]:,} filas")

Merge pediátricos: 6,478 filas


In [17]:
# Merge con dataset adultos
df_merged_adultos = pd.merge(
    df_anomalias_adultos[['Documento PMD', 'es_anomalia_adultos']],
    df_target[['Documento PMD', 'target']],
    on='Documento PMD',
    how='inner'
)

print(f"Merge adultos: {df_merged_adultos.shape[0]:,} filas")

Merge adultos: 23,387 filas


In [19]:
# Crear dataset combinado con todas las anomalías
df_combinado = df_target[['Documento PMD', 'target']].copy()

# Agregar columnas de anomalías de cada dataset
df_combinado = pd.merge(
    df_combinado,
    df_anomalias_completo[['Documento PMD', 'es_anomalia_completo']],
    on='Documento PMD',
    how='left'
)

df_combinado = pd.merge(
    df_combinado,
    df_anomalias_pediatricos[['Documento PMD', 'es_anomalia_pediatricos']],
    on='Documento PMD',
    how='left'
)

df_combinado = pd.merge(
    df_combinado,
    df_anomalias_adultos[['Documento PMD', 'es_anomalia_adultos']],
    on='Documento PMD',
    how='left'
)

# Rellenar NaN con 0 (si no está en algún dataset, no es anomalía en ese grupo)
df_combinado['es_anomalia_completo'] = df_combinado['es_anomalia_completo'].fillna(0).astype(int)
df_combinado['es_anomalia_pediatricos'] = df_combinado['es_anomalia_pediatricos'].fillna(0).astype(int)
df_combinado['es_anomalia_adultos'] = df_combinado['es_anomalia_adultos'].fillna(0).astype(int)

print(f"Dataset combinado: {df_combinado.shape[0]:,} filas × {df_combinado.shape[1]} columnas")

Dataset combinado: 29,865 filas × 5 columnas


## 5. Estadísticas Descriptivas

In [20]:
# Distribución de target
print("=" * 80)
print("DISTRIBUCIÓN DE TARGET")
print("=" * 80)

target_counts = df_combinado['target'].value_counts()
print(f"\nTarget = 0 (No necesitaba valoración): {target_counts.get(0, 0):,} ({target_counts.get(0, 0)/len(df_combinado)*100:.2f}%)")
print(f"Target = 1 (Necesitaba valoración): {target_counts.get(1, 0):,} ({target_counts.get(1, 0)/len(df_combinado)*100:.2f}%)")

DISTRIBUCIÓN DE TARGET

Target = 0 (No necesitaba valoración): 12,390 (41.49%)
Target = 1 (Necesitaba valoración): 17,475 (58.51%)


In [21]:
# Distribución de anomalías por grupo
print("\n" + "=" * 80)
print("DISTRIBUCIÓN DE ANOMALÍAS POR GRUPO")
print("=" * 80)

for col in ['es_anomalia_completo', 'es_anomalia_pediatricos', 'es_anomalia_adultos']:
    grupo_nombre = col.replace('es_anomalia_', '').upper()
    anomalias_counts = df_combinado[col].value_counts()
    
    print(f"\n{grupo_nombre}:")
    print(f"  No anomalía (0): {anomalias_counts.get(0, 0):,} ({anomalias_counts.get(0, 0)/len(df_combinado)*100:.2f}%)")
    print(f"  Anomalía (1): {anomalias_counts.get(1, 0):,} ({anomalias_counts.get(1, 0)/len(df_combinado)*100:.2f}%)")


DISTRIBUCIÓN DE ANOMALÍAS POR GRUPO

COMPLETO:
  No anomalía (0): 14,923 (49.97%)
  Anomalía (1): 14,942 (50.03%)

PEDIATRICOS:
  No anomalía (0): 26,624 (89.15%)
  Anomalía (1): 3,241 (10.85%)

ADULTOS:
  No anomalía (0): 18,214 (60.99%)
  Anomalía (1): 11,651 (39.01%)


## 6. Tablas de Contingencia

In [22]:
print("=" * 80)
print("TABLAS DE CONTINGENCIA: TARGET vs ANOMALÍAS")
print("=" * 80)

for col_anomalia in ['es_anomalia_completo', 'es_anomalia_pediatricos', 'es_anomalia_adultos']:
    grupo_nombre = col_anomalia.replace('es_anomalia_', '').upper()
    print(f"\n{grupo_nombre}:")
    print("-" * 60)
    
    # Tabla de contingencia absoluta
    tabla = pd.crosstab(
        df_combinado['target'],
        df_combinado[col_anomalia],
        margins=True,
        margins_name="Total"
    )
    tabla.index = ['Target=0 (No valoración)', 'Target=1 (Valoración)', 'Total']
    tabla.columns = ['No Anomalía', 'Anomalía', 'Total']
    print("\nFrecuencias absolutas:")
    print(tabla)
    
    # Porcentajes por fila
    tabla_pct_fila = pd.crosstab(
        df_combinado['target'],
        df_combinado[col_anomalia],
        normalize='index'
    ) * 100
    tabla_pct_fila.index = ['Target=0', 'Target=1']
    tabla_pct_fila.columns = ['No Anomalía', 'Anomalía']
    print("\nPorcentajes por fila (Target):")
    print(tabla_pct_fila.round(2))
    
    # Porcentajes por columna
    tabla_pct_col = pd.crosstab(
        df_combinado['target'],
        df_combinado[col_anomalia],
        normalize='columns'
    ) * 100
    tabla_pct_col.index = ['Target=0', 'Target=1']
    tabla_pct_col.columns = ['No Anomalía', 'Anomalía']
    print("\nPorcentajes por columna (Anomalías):")
    print(tabla_pct_col.round(2))
    print("\n" + "=" * 80)

TABLAS DE CONTINGENCIA: TARGET vs ANOMALÍAS

COMPLETO:
------------------------------------------------------------

Frecuencias absolutas:
                          No Anomalía  Anomalía  Total
Target=0 (No valoración)         6477      5913  12390
Target=1 (Valoración)            8446      9029  17475
Total                           14923     14942  29865

Porcentajes por fila (Target):
          No Anomalía  Anomalía
Target=0        52.28     47.72
Target=1        48.33     51.67

Porcentajes por columna (Anomalías):
          No Anomalía  Anomalía
Target=0         43.4     39.57
Target=1         56.6     60.43


PEDIATRICOS:
------------------------------------------------------------

Frecuencias absolutas:
                          No Anomalía  Anomalía  Total
Target=0 (No valoración)        10913      1477  12390
Target=1 (Valoración)           15711      1764  17475
Total                           26624      3241  29865

Porcentajes por fila (Target):
          No Anomalía  Ano

## 7. Exportación de Datasets

In [23]:
# Exportar datasets individuales mergeados
df_merged_completo.to_csv('../../OPERA_merged_target_anomalias_completo.csv', index=False)
df_merged_pediatricos.to_csv('../../OPERA_merged_target_anomalias_pediatricos.csv', index=False)
df_merged_adultos.to_csv('../../OPERA_merged_target_anomalias_adultos.csv', index=False)
df_combinado.to_csv('../../OPERA_merged_target_anomalias_combinado.csv', index=False)

print("=" * 80)
print("EXPORTACIÓN COMPLETADA")
print("=" * 80)
print("\n✓ Datasets exportados:")
print("  - OPERA_merged_target_anomalias_completo.csv")
print("  - OPERA_merged_target_anomalias_pediatricos.csv")
print("  - OPERA_merged_target_anomalias_adultos.csv")
print("  - OPERA_merged_target_anomalias_combinado.csv (todos los grupos combinados)")

EXPORTACIÓN COMPLETADA

✓ Datasets exportados:
  - OPERA_merged_target_anomalias_completo.csv
  - OPERA_merged_target_anomalias_pediatricos.csv
  - OPERA_merged_target_anomalias_adultos.csv
  - OPERA_merged_target_anomalias_combinado.csv (todos los grupos combinados)
